In [ ]:
%%writefile chatbot-streamlit.py
import oracledb
import oci
from langchain.memory import ConversationBufferMemory
from langchain.chains import ConversationalRetrievalChain
from langchain_community.vectorstores.utils import DistanceStrategy
from langchain_community.vectorstores.oraclevs import OracleVS
from langchain_community.chat_models.oci_generative_ai import ChatOCIGenAI
from langchain_community.embeddings import OCIGenAIEmbeddings
from LoadProperties import LoadProperties

print("Successfully imported libraries and modules")

# Setup basic variables
properties = LoadProperties()

#Declare username and password and dsn (data connection string) - Copy from the activity guide.
un = "<username>"
pw = "<password>"
cs = "localhost/FREEPDB1"

# Connect to the database
try:
    conn23c = oracledb.connect(user=un, password=pw, dsn=cs)
    print("Connection successful!")
except Exception as e:
    print("Connection failed!")

#In this demo we will explore using ConversationalRetrievalChain chain to retrieve relevant documents and send these as a context in a query.
# We will use Chroma vectorstore.

#Step 1 - here we create a function that creates a chain

def create_chain():
    embed_model = OCIGenAIEmbeddings(
        model_id=properties.getEmbeddingModelName(),
        service_endpoint=properties.getEndpoint(),
        compartment_id=properties.getCompartment(),
        auth_type="INSTANCE_PRINCIPAL",
    )

    vs = OracleVS(embedding_function=embed_model, client=conn23c, table_name="DEMO_TABLE",
                  distance_strategy=DistanceStrategy.DOT_PRODUCT)

    retv = vs.as_retriever(search_type="similarity", search_kwargs={'k': 3})

    llm = ChatOCIGenAI(
        model_id=properties.getModelName(),
        service_endpoint=properties.getEndpoint(),
        compartment_id=properties.getCompartment(),
        auth_type="INSTANCE_PRINCIPAL",
        model_kwargs={"max_tokens":200}
        )
    memory = ConversationBufferMemory(llm=llm, memory_key="chat_history", return_messages=True, output_key='answer')
    qa = ConversationalRetrievalChain.from_llm(llm, retriever=retv, memory=memory, return_source_documents=True)
    return qa


#Step 2 - here we declare a chat function
def chat(llm_chain, user_input):
    # generate a prediction for a prompt
    bot_json = llm_chain.invoke(user_input)
    return {"bot_response": bot_json}

#Step 3 - here we setup Streamlit text input and pass input text to chat function.
# chat function returns the response and we print it.

if __name__ == "__main__":
    import streamlit as st

    st.subheader("OU Chatbot")
    col1, col2 = st.columns([4,1])

    def initialize_session_state():
        if "llm_chain" not in st.session_state:
            st.session_state["llm_chain"] = create_chain()
            llm_chain = st.session_state["llm_chain"]
        else:
            llm_chain = st.session_state["llm_chain"]
        return llm_chain

    user_input = st.chat_input()
    with col1:
        col1.subheader("----Ask me about OCI AI Foundations course----")
        if "messages" not in st.session_state:
            st.session_state.messages = []
        if user_input:
            llm_chain = initialize_session_state()
            bot_response = chat(llm_chain, user_input)
            print("bot_response->\n", bot_response)
            st.session_state.messages.append({"role" : "chatbot", "content" : bot_response})
            #st.write("OU Assistant Response: ", bot_response)
            for message in st.session_state.messages:
                st.chat_message("user")
                st.write("Question: ", message['content']['bot_response']['question'])
                st.chat_message("assistant")
                st.write("Answer: ", message['content']['bot_response']['answer'])
                st.chat_message("assistant")
                for doc in message['content']['bot_response']['source_documents']:
                    st.write("Chunk Id Is: ", doc.metadata['id'] + "  \n"+ "-page->"+str(doc.metadata['link']))                    

In [ ]:
!streamlit run chatbot-streamlit.py --server.address 127.0.0.1 --server.port 8501